In [ ]:
import jax
import jax.numpy as jnp
import plotly.express as px
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np

In [ ]:
KEY = jax.random.PRNGKey(11)
def random(*shape):
    global KEY
    KEY, subkey = jax.random.split(KEY)
    return jax.random.normal(subkey, shape)

def symMat(dim):
    A = jnp.array(random(dim, dim) > 0.2, jnp.float32)
    return (A - A.T)

def scale_free_antisymmetric(degree, dim):
    global KEY
    KEY, subkey = jax.random.split(KEY)
    """Build negative graph-Laplacian dynamics from a Watts-Strogatz graph."""
    if degree >= dim:
        raise ValueError(
            "small_world_degree must be smaller than dim for the small-world generator"
        )
    graph_seed = int(jax.random.bits(subkey, (), dtype=jnp.uint32))
    graph = nx.scale_free_graph(n=dim, seed=graph_seed)
    adjacency = nx.to_numpy_array(graph, dtype=np.float32)

    dense_matrix = (adjacency - adjacency.T)

    return adjacency

In [ ]:
def get_synced(systemA, systemB, sync_over):
    def foo(stateA, stateB, intensivity):
        s1 = systemA(stateA)
        s2 = systemB(stateB)
        for sync in sync_over:
            path1, path2, w = sync
            s_v = s1[path1]*w + s2[path2]*(1 - w)
            b, a = 1 - intensivity, intensivity
            s1 = s1.at[path1].set(s_v*a + s1[path1]*b)
            s2 = s2.at[path2].set(s_v*a + s2[path2]*b)
        return s1, s2

    return foo

In [ ]:
def get_transformation(binFoo, adj1, adj2):
    def transform_state(state):
        y = state
        for _ in range(1):
            y = binFoo(adj1 @ state, adj2 @ y)
        return y
    return transform_state

def binFoo(x, y):
    return (-(x**2 + y**2 - 1)*y*0.1 + y)
    return (-(y**2 - x)*y*0.01 + y) % 60.0
    # return ((x - y)*0.01 + y)


def generate_circle(dim):
    A = jnp.eye(dim)
    Q = A.at[jnp.arange(-1, A.shape[0] - 1)].get()
    Q1 = jnp.ones((dim, dim))/dim

    return get_transformation(binFoo, Q, Q1), Q, Q1

In [ ]:
dim = 9
dim1 = dim
dim2 = dim
system1, Q11, Q12 = generate_circle(dim1)
system2, Q21, Q22 = generate_circle(dim2)

s1i, s2i = random(dim1)*3, random(dim2)*0.1
while abs(s1i - s2i).min() < 0.5:
    s1i, s2i = random(dim1)*3, random(dim2)*0.1

In [ ]:
abs(s1i - s2i).min()

In [ ]:
G = nx.from_numpy_array(Q11 + Q12)
G1 = nx.from_numpy_array(Q21 + Q22)
pos = nx.layout.kamada_kawai_layout(G)
pos1 = nx.layout.kamada_kawai_layout(G1)

nx.draw(G, pos, node_size=5)
plt.show()
nx.draw(G1, pos1, node_size=5)
plt.show()

In [ ]:
s = get_synced(system1, system2, [[0, 0, 1.0]])
H = []
H1, H2 = [], []
d = []
s1, s2 = s1i, s2i

assert abs(s1 - s2).sum() > 0.1

for i in range(1000):
    #if 500 < i < 700:
    s1, s2 = s(s1, s2, 500 < i < 700)
    #else:
    # s1, s2 = system1(s1), system2(s2)
    if (i % 10 == 0):
        m = jnp.mean((s1 - s2)**2) - (s1[0] - s2[0])**2/len(s1)
        H.append(m)
        H1.append(s1)
        H2.append(s2)
        d.append(s1 - s2)

In [ ]:
start = 0
plt.plot(H[start:])
plt.show()

plt.plot(H1[start:])
plt.show()

plt.plot(H2[start:])
plt.show()

plt.plot(d[start:])
plt.show()